<a href="https://colab.research.google.com/github/AJDLGM/Equipo1_601/blob/main/shoppilot_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ShopPilot - customer service chatbot (Version 2)

You ask as the customer, the bot answers from the company sources. Updated for policy **CS-2026-05** (Source E, approved 12 May 2026).

**Step 1:** run the next cell and upload the 5 files from `data_v2`: `orders_v2.csv`, `products.json`, `support_notes_v2.json`, `faq_v2.txt`, `policy_E.json`.

**Step 2:** run the remaining cells. Original files are never modified.

In [2]:
# Step 1 - upload the data_v2 files (skip if they are already in /content)
try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print('Not in Colab: put the files next to this notebook.')

Saving policy_E.json to policy_E (1).json
Saving orders_v2.csv to orders_v2 (1).csv
Saving products.json to products (1).json
Saving faq_v2.txt to faq_v2 (1).txt
Saving support_notes_v2.json to support_notes_v2 (1).json


In [3]:
"""ShopPilot - customer service chatbot prototype (Version 2).

The CUSTOMER asks, the BOT answers. Every answer is built only from the sources:
  A orders (orders_v2.csv)          snapshot 12 May 2026
  B product catalogue (products.json) snapshot 12 May 2026
  C FAQ (faq_v2.txt)                update date unknown  -> weak evidence
  D support notes (support_notes_v2.json) unapproved     -> never used as evidence
  E policy message CS-2026-05 (policy_E.json) approved 12 May 2026 -> overrides C for returns

Source priority:  E (approved policy)  >  A, B (snapshots)  >  C (undated)  >  D (not evidence)
Missing data is never invented: the bot says what is missing and hands over to a human.
"""
import csv, json, re
from datetime import date, datetime, timedelta
from pathlib import Path

# ---------------------------------------------------------------- 1. Load data
FILES = ["orders_v2.csv", "products.json", "support_notes_v2.json", "faq_v2.txt", "policy_E.json"]

def find_dir():
    for d in [Path("/content/data_v2"), Path("/content"), Path("data_v2"), Path(".")]:
        if all((d / f).exists() for f in FILES):
            return d
    raise FileNotFoundError("Upload these files to Colab (/content): " + ", ".join(FILES))

D = find_dir()
ORDERS = {r["order_id"]: r for r in csv.DictReader(open(D / "orders_v2.csv", encoding="utf-8"))}
PRODUCTS = {p["product_id"]: p for p in json.load(open(D / "products.json", encoding="utf-8"))}
NOTES = json.load(open(D / "support_notes_v2.json", encoding="utf-8"))   # loaded for the record, NOT used as evidence
FAQ = open(D / "faq_v2.txt", encoding="utf-8").read()
POLICY_E = json.load(open(D / "policy_E.json", encoding="utf-8"))

CUTOFF = date.fromisoformat(POLICY_E["cutoff_date"])         # 2026-05-01
NEW_RULE, OLD_RULE = POLICY_E["rules"][0], POLICY_E["rules"][1]
SCENARIO_TODAY = date(2026, 5, 12)    # workbook: the scenario snapshot is 12 May 2026 regardless of real date
USE_POLICY_E = True                   # False reproduces Version 1 behaviour (FAQ only) for before/after tests

DELIVERY_ESTIMATE = "two to four business days" if "two to four business days" in FAQ else None

# ---------------------------------------------------------------- 2. Helpers
MONTHS = {m: i for i, m in enumerate(
    ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october", "november", "december"], 1)}

def parse_dates(text):
    found = []
    for m in re.finditer(r"\b(\d{4})-(\d{2})-(\d{2})\b", text):
        try: found.append(date(int(m[1]), int(m[2]), int(m[3])))
        except ValueError: pass
    for m in re.finditer(r"\b(\d{1,2})(?:st|nd|rd|th)?\s+(january|february|march|april|may|june|july|august|september|october|november|december)(?:\s+(\d{4}))?", text, re.I):
        try: found.append(date(int(m[3] or 2026), MONTHS[m[2].lower()], int(m[1])))
        except ValueError: pass
    return found

def fmt(d): return f"{d.day} {d.strftime('%B %Y')}"

def policy_for(order_date):
    """Source E: the ORDER DATE (purchase date) selects the policy."""
    if not USE_POLICY_E:   # Version 1: FAQ only
        return {"days": 30, "fee": 0.0, "label": "FAQ (Source C)", "source": ["C"]}
    if order_date >= CUTOFF:
        return {"days": NEW_RULE["return_days"], "fee": NEW_RULE["return_shipping_eur"], "label": "policy CS-2026-05", "source": ["E"]}
    return {"days": OLD_RULE["return_days"], "fee": OLD_RULE["return_shipping_eur"], "label": "previous voluntary policy", "source": ["E"]}

def fee_text(p): return "free of charge" if p["fee"] == 0 else f"EUR {p['fee']:.2f} per return shipment"

PRODUCT_ALIASES = {"citylight": "P10", "city light": "P10", "tourbag": "P11", "tour bag": "P11", "minipump": "P12", "mini pump": "P12"}
VALVES = ["presta", "schrader", "dunlop", "woods", "sclaverand"]

HANDOVER = "I'll pass this to a colleague from our service team who can check it properly."

def reply(text, sources, handover=False, evidence=None):
    return {"answer": text, "sources": sources, "handover": handover, "evidence": evidence or []}

# ---------------------------------------------------------------- 3. Topics
def a_return(q, order, delivery):
    if not order:
        if re.search(r"\bB\d{3}\b", q, re.I):
            return reply("I can't find that order number in our order data. " + HANDOVER, ["A"], True)
        if not USE_POLICY_E:
            return reply("Happy to help. Our voluntary return period is 30 days from delivery and return shipping is free of charge.", ["C"])
        return reply(
            "Happy to help. Which order is it? Please send me your order number (for example B101), because the rules depend on the order date.\n"
            f"- Orders placed on or after 1 May 2026: {NEW_RULE['return_days']} calendar days from delivery, return shipping EUR {NEW_RULE['return_shipping_eur']:.2f} per return shipment.\n"
            f"- Orders placed before 1 May 2026: {OLD_RULE['return_days']} days from delivery, free return shipping.\n"
            "The period always starts on delivery, not on purchase.", ["E"])
    try: od = date.fromisoformat(order["order_date"])
    except ValueError: return reply("The order date of this order is missing or unreadable, so I can't tell which return policy applies. " + HANDOVER, ["A"], True)
    p = policy_for(od); st = order["shipping_status"]
    asks_cost = bool(re.search(r"cost|price|pay|fee|free|shipping|postage|how much", q, re.I))
    head = (f"Your order {order['order_id']} was placed on {fmt(od)}, so {p['label']} applies: "
            f"you can return it voluntarily within {p['days']} calendar days of delivery, and return shipping is {fee_text(p)}.")
    ev = [f"order_date {order['order_date']} -> {p['label']}", f"shipping_status {st}"]
    if asks_cost and not re.search(r"how long|days|period|deadline|still|until|when", q, re.I):
        head = f"For order {order['order_id']} (placed {fmt(od)}), return shipping is {fee_text(p)} ({p['label']}). The return period is {p['days']} calendar days from delivery."
    if delivery:
        if delivery < od:
            return reply(head + f" However, the delivery date you gave ({fmt(delivery)}) is before the order date, so something doesn't add up. " + HANDOVER, p["source"] + ["A"], True, ev)
        last = delivery + timedelta(days=p["days"])
        left = (last - SCENARIO_TODAY).days
        when = f"That makes {fmt(last)} the last day to return it." + (f" As of our data date ({fmt(SCENARIO_TODAY)}) you still have {left} days." if left >= 0 else f" As of our data date ({fmt(SCENARIO_TODAY)}) that period has passed.")
        extra = ""
        if st != "delivered":
            extra = f" Note: our system shows this order as '{st}', not delivered, so please confirm the delivery date with us."
        return reply(head + f" You told me it arrived on {fmt(delivery)}. " + when + extra, p["source"] + ["A"], st == "unknown", ev + [f"customer-stated delivery {delivery}"])
    if st == "delivered":
        tail = " Our data does not contain the delivery date, so I can't tell you the exact last day. If you tell me the date it arrived, I can work it out."
    elif st in ("in_transit", "pending"):
        tail = f" Your order is currently '{st}', so the return period has not started yet; it begins on delivery."
    else:
        tail = " The shipping status of this order is unknown in our data, so I can't confirm whether or when it was delivered. " + HANDOVER
    return reply(head + tail, p["source"] + ["A"], st == "unknown", ev)

def a_delivery(q, order):
    if not order:
        return reply("Which order do you mean? Please send me your order number (for example B102). I should say up front that I don't have delivery dates or parcel tracking, only the shipping status.", ["A"])
    st = order["shipping_status"]
    est = f" As a general guide only, our FAQ says delivery normally takes {DELIVERY_ESTIMATE}, but that is not a promise for your parcel and I can't confirm it applies here." if DELIVERY_ESTIMATE else ""
    base = {"delivered": "It is shown as delivered", "in_transit": "It is shown as in transit", "pending": "It is shown as pending (not shipped yet)"}.get(st)
    if st == "unknown":
        return reply(f"I can't give you an arrival date for {order['order_id']}: I have no delivery dates or tracking, and its shipping status is unknown in our data. " + HANDOVER, ["A"], True)
    return reply(f"I can't give you an exact arrival date for {order['order_id']} because our data contains no delivery dates or parcel tracking. {base}." + (est if st != "delivered" else ""), ["A"] + (["C"] if est and st != "delivered" else []))

def a_status(q, order):
    if not order:
        return reply("Please send me your order number (for example B101) and I'll check its shipping status.", ["A"])
    st = order["shipping_status"]
    if st == "unknown":
        return reply(f"The shipping status of {order['order_id']} is unknown in our data, so I can't confirm where it is. " + HANDOVER, ["A"], True)
    txt = {"delivered": "delivered", "in_transit": "in transit", "pending": "pending, so it hasn't shipped yet"}[st]
    return reply(f"According to our order data (snapshot {fmt(SCENARIO_TODAY)}), order {order['order_id']} is {txt}. I don't have a delivery date or parcel tracking.", ["A"])

def a_product(q, order, prod, pid):
    if pid and prod is None:
        return reply(f"I can't find the product on this order in our catalogue, so I can't answer product questions about it. " + HANDOVER, ["A", "B"], True)
    if not prod:
        return reply("Which product do you mean? I can answer questions about CityLight, TourBag and MiniPump.", ["B"])
    name, out, src, ho = prod["name"], [], ["B"], False
    ql = q.lower()
    if re.search(r"valve|presta|schrader|dunlop|woods|sclaverand|pump", ql):
        if "supported_valves" not in prod:
            out.append(f"Our catalogue has no valve information for {name}.")
        else:
            asked = [v for v in VALVES if v in ql]
            lst = " and ".join(prod["supported_valves"])
            if not asked: out.append(f"{name} supports {lst} valves.")
            elif asked[0] in [s.lower() for s in prod["supported_valves"]]: out.append(f"Yes, {name} supports {asked[0].title()} valves (catalogue: {lst}).")
            else:
                out.append(f"I can't confirm that {name} works with {asked[0].title()} valves: our catalogue lists only {lst}. I don't want to promise something we haven't confirmed. " + HANDOVER); ho = True
    if re.search(r"charg|usb|cable|port|plug", ql):
        out.append(f"{name} charges via {prod['charging_port']}." if "charging_port" in prod else f"Our catalogue has no charging information for {name}.")
    if re.search(r"capacity|litre|liter|volume|how much (?:fit|hold)|\bhold|\bfit", ql):
        out.append(f"{name} has a capacity of {prod['capacity_litres']} litres." if "capacity_litres" in prod else f"Our catalogue has no capacity information for {name}.")
    if re.search(r"rain|water|wet|splash|storm|shower|weather|submerge", ql):
        if "water_protection" not in prod: out.append(f"Our catalogue has no water-protection information for {name}.")
        else:
            lvl = prod["water_protection"]
            heavy = re.search(r"heavy|downpour|storm|pour|all day|torrential", ql)
            if lvl == "waterproof": out.append(f"Our catalogue lists {name} as waterproof.")
            elif heavy:
                out.append(f"Our catalogue lists {name} only as {lvl}. That is not enough for me to say it is suitable for heavy rain, so I can't promise that. " + HANDOVER); ho = True
            else: out.append(f"Our catalogue lists {name} as {lvl}. It doesn't say how much rain it can take.")
    if not out:
        return reply(f"I can tell you about {name}'s " + ", ".join(k.replace('_', ' ') for k in prod if k not in ('product_id', 'name')) + ". What would you like to know?", ["B"])
    return reply(" ".join(out), src, ho)

# ---------------------------------------------------------------- 4. Chatbot
class ShopPilot:
    def __init__(self): self.order = None; self.product = None
    def ask(self, q):
        m = re.search(r"\bB\d{3}\b", q, re.I)
        if m: self.order = ORDERS.get(m[0].upper()); self.last_order_id = m[0].upper()
        order = self.order if (m or self.order) else None
        pid = None; prod = None
        for alias, ppid in PRODUCT_ALIASES.items():
            if alias in q.lower(): pid, prod = ppid, PRODUCTS[ppid]; self.product = (pid, prod)
        if not pid and order and re.search(r"it|this|my|order|B\d{3}", q, re.I) and not self._product_free(q):
            pid = order["product_id"]; prod = PRODUCTS.get(pid)
        if not pid and not order and self.product: pid, prod = self.product
        delivery = next((d for d in parse_dates(q)), None)
        ql = q.lower()
        if re.fullmatch(r"\s*(hi|hello|hey|hola)[!. ]*", ql):
            return reply("Hello! I'm ShopPilot. I can help with order status, product questions (CityLight, TourBag, MiniPump) and voluntary returns.", [])
        parts = []
        if re.search(r"return|refund|send (?:it )?back|money back|exchange", ql): parts.append(a_return(q, order, delivery))
        elif re.search(r"arrive|arrival|when (?:will|does|is)|delivery date|how long|which date|what date|eta", ql): parts.append(a_delivery(q, order))
        elif m and re.search(r"status|where|track|shipped|delivered|stand|update", ql): parts.append(a_status(q, order))
        if re.search(r"valve|presta|schrader|dunlop|charg|usb|cable|port\b|capacity|litre|liter|volume|rain|water|splash|wet|waterproof|storm|hold|fit\b", ql) and not parts:
            parts.append(a_product(q, order, prod, pid))
        if not parts and m: parts.append(a_status(q, order))
        if not parts:
            return reply("I can help with order status, product questions (CityLight, TourBag, MiniPump) and voluntary returns, but I don't have information on that. " + HANDOVER, [], True)
        r = parts[0]
        for extra in parts[1:]:
            r = reply(r["answer"] + "\n" + extra["answer"], sorted(set(r["sources"] + extra["sources"])), r["handover"] or extra["handover"])
        return r
    @staticmethod
    def _product_free(q): return bool(re.search(r"return|refund|arrive|deliver|status|where|track", q, re.I))

## Example questions
Before/after the policy change (set `USE_POLICY_E = False` to see Version 1 behaviour).

In [4]:
for q in ["Can I return order B101? How long do I have?",
          "How much does return shipping cost for order B102?",
          "Order B101 was delivered on 2026-05-10. Can I still return it?",
          "I want to return my order.",
          "On which date will order B102 arrive?",
          "Can I use MiniPump with a Dunlop valve?"]:
    r = ShopPilot().ask(q)
    print("You:", q); print("ShopPilot:", r["answer"]); print("  [Sources:", ", ".join(r["sources"]) or "-", "| handover:", r["handover"], "]\n")

You: Can I return order B101? How long do I have?
ShopPilot: Your order B101 was placed on 3 May 2026, so policy CS-2026-05 applies: you can return it voluntarily within 14 calendar days of delivery, and return shipping is EUR 4.90 per return shipment. Our data does not contain the delivery date, so I can't tell you the exact last day. If you tell me the date it arrived, I can work it out.
  [Sources: E, A | handover: False ]

You: How much does return shipping cost for order B102?
ShopPilot: For order B102 (placed 4 May 2026), return shipping is EUR 4.90 per return shipment (policy CS-2026-05). The return period is 14 calendar days from delivery. Your order is currently 'in_transit', so the return period has not started yet; it begins on delivery.
  [Sources: E, A | handover: False ]

You: Order B101 was delivered on 2026-05-10. Can I still return it?
ShopPilot: Your order B101 was placed on 3 May 2026, so policy CS-2026-05 applies: you can return it voluntarily within 14 calendar day

## Chat with ShopPilot
Ask about an order (B101-B105), a product (CityLight, TourBag, MiniPump) or a return. Type `quit` to stop.

In [ ]:
# ---------------------------------------------------------------- 5. Chat loop
def chat():
    bot = ShopPilot()
    print("ShopPilot: Hello! Ask me about an order, a product (CityLight, TourBag, MiniPump) or a return. Type 'quit' to stop.")
    while True:
        q = input("\nYou: ").strip()
        if q.lower() in ("quit", "exit"): print("ShopPilot: Goodbye!"); break
        if not q: continue
        r = bot.ask(q)
        print("ShopPilot:", r["answer"])
        print(f"   [Sources: {', '.join(r['sources']) or '-'}{' | handed over to human' if r['handover'] else ''}]")

chat()

ShopPilot: Hello! Ask me about an order, a product (CityLight, TourBag, MiniPump) or a return. Type 'quit' to stop.
ShopPilot: Happy to help. Which order is it? Please send me your order number (for example B101), because the rules depend on the order date.
- Orders placed on or after 1 May 2026: 14 calendar days from delivery, return shipping EUR 4.90 per return shipment.
- Orders placed before 1 May 2026: 30 days from delivery, free return shipping.
The period always starts on delivery, not on purchase.
   [Sources: E]
ShopPilot: According to our order data (snapshot 12 May 2026), order B101 is delivered. I don't have a delivery date or parcel tracking.
   [Sources: A]
ShopPilot: I can help with order status, product questions (CityLight, TourBag, MiniPump) and voluntary returns, but I don't have information on that. I'll pass this to a colleague from our service team who can check it properly.
   [Sources: - | handed over to human]
ShopPilot: The shipping status of B105 is unknown i